### **Incremental Data Ingestion**

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.raw.gold;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.gold.goldVolume

In [0]:
dbutils.widgets.text("src","")

In [0]:
src_value = dbutils.widgets.get("src")
src_value

In [0]:
# 1. Define paths
source_path = f"/Volumes/workspace/raw/rawvolume/rawdata/{src_value}"

schema_path = f"/Volumes/workspace/bronze/bronzevolume/{src_value}/schema"

checkpoint_path = f"/Volumes/workspace/bronze/bronzevolume/{src_value}/checkpoint"

output_path =f"/Volumes/workspace/bronze/bronzevolume/{src_value}/data"


# 2. Read source data using Auto Loader (streaming)
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .load(source_path)
)


# 3. Write streaming data to Bronze in Delta format
query = (
    df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(once=True)
        .option("checkpointLocation", checkpoint_path)
        .option("path", output_path)
        .start()
)

# 4. Wait for the streaming query to finish
query.awaitTermination()

print("Bronze streaming ingestion completed.")

# 5. Display the Bronze data
display(spark.read.format("delta").load(output_path))

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/bronze/bronzevolume/bookings/data/`